In [1]:
import transformers
import accelerate

print(transformers.__version__)
print(accelerate.__version__)

5.13.0
1.14.0


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv(r"C:\Users\EileneAnnaKuriakose\Desktop\IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["sentiment"])

In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [6]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [7]:
from transformers import ElectraTokenizer

tokenizer = ElectraTokenizer.from_pretrained(
    "google/electra-small-discriminator"
)

In [8]:
def tokenize_function(examples):
    return tokenizer(
        examples["review"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

In [9]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [10]:
from transformers import ElectraForSequenceClassification

model = ElectraForSequenceClassification.from_pretrained(
    "google/electra-small-discriminator",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: google/electra-small-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstre

In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

In [12]:
from sklearn.metrics import accuracy_score
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}

In [20]:
from transformers import Trainer, TrainingArguments



In [21]:
train_dataset = train_dataset.select(range(100))
test_dataset = test_dataset.select(range(20))
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.413088,0.850000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7, training_loss=0.30552925382341656, metrics={'train_runtime': 23.6835, 'train_samples_per_second': 4.222, 'train_steps_per_second': 0.296, 'total_flos': 1470982041600.0, 'train_loss': 0.30552925382341656, 'epoch': 1.0})

In [22]:
trainer.save_model("./electra_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
tokenizer.save_pretrained("./electra_model")

('./electra_model\\tokenizer_config.json', './electra_model\\tokenizer.json')

In [24]:
from transformers import ElectraForSequenceClassification, ElectraTokenizer

loaded_model = ElectraForSequenceClassification.from_pretrained("./electra_model")
loaded_tokenizer = ElectraTokenizer.from_pretrained("./electra_model")

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

In [25]:
import torch

text = "This movie was absolutely amazing. I loved every scene."

inputs = loaded_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

with torch.no_grad():
    outputs = loaded_model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1)

if prediction.item() == 1:
    print("Prediction: Positive 😊")
else:
    print("Prediction: Negative 😞")

Prediction: Positive 😊


In [26]:
text = "This movie was boring and a complete waste of time."

inputs = loaded_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

with torch.no_grad():
    outputs = loaded_model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1)

if prediction.item() == 1:
    print("Prediction: Positive 😊")
else:
    print("Prediction: Negative 😞")

Prediction: Negative 😞


In [27]:
text = "This movie was fantastic. I really enjoyed watching it."

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits)

print("Prediction:", "Positive" if prediction==1 else "Negative")

Prediction: Positive
